# Case study: a fishery and its fleet {#sec-case-fishery}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c16-fishery.ipynb)

A coastal region is about to reopen its fishery for a demersal stock, fish that live and feed near the sea bed, after a ten-year closure. The stock has rebuilt to about 90% of its unfished size. The last time the fishery was open, landings boomed, the fleet grew more than tenfold and then the fish were gone; boats bought in the good years were still being paid off when there was nothing left to catch. The regional fisheries authority asks three questions: why did it happen, will it happen again, and which rules would stop it?

This chapter answers them by running the whole modelling process of the book on one problem, compressed into one chapter: the problem and its reference mode, a dynamic hypothesis and a boundary, the model built step by step, tests, an explanation of the behaviour, policy experiments with a robustness check, and a brief for the decision maker. The fishery is generic, not a model of any real fishery, and its records are synthetic. The sketch after the setup cell shows the structure we will build.

**Who it is for.** Analysts who want to see the full process applied to a renewable resource, and anyone who models a stock that regrows (a forest, an aquifer, grazing land) together with the capital that exploits it. **Who can skip.** Readers who know the economics of fisheries can skim sections 1 and 2 and start at section 3. **Prerequisites.** Chapter 5, Loops that grow, settle and tip (logistic growth); chapter 6, Delays, pipelines and memory (ageing chains and smoothing); chapter 8, Why things swing (supply lines and pricing); chapter 9, Patterns that repeat (the tragedy of the commons); chapter 12, Testing your model; and chapter 15, From model to decision (policies as parameters, scenario grids). **Time.** About three hours with the exercises.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package,
# which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 16
rng = np.random.default_rng(SEED)

# Okabe-Ito colours with a different line style each, readable in greyscale
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
             "#E69F00", "#56B4E9", "#000000", "#F0E442"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)),
          (0, (1, 3)), (0, (8, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__)

In [ ]:
#| fig-alt: "Stock-and-flow sketch of the fishery. Regrowth fills the Fish stock and the Catch drains it. Below, boats move through a chain: Ordering fills Boats on order, Building moves them to New boats, Ageing to Old boats, and Scrapping removes them. Dashed arrows show the fleet setting the catch, the stock setting the catch per boat, and profit driving orders and early scrapping."
from matplotlib.patches import Circle, FancyBboxPatch


def draw_sfd(ax, stocks, flows, links):
    """A small stock-and-flow diagram (as in chapters 8 and 9)."""
    w, h = 1.7, 0.8
    for name, (x, y) in stocks.items():
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                    boxstyle="round,pad=0.02", fc="white",
                                    ec="black", lw=1.5))
        ax.text(x, y, name, ha="center", va="center", fontsize=9)

    def inside(p):
        return any(abs(p[0] - x) <= w / 2 + 1e-9
                   and abs(p[1] - y) <= h / 2 + 1e-9
                   for x, y in stocks.values())

    for label, a, b, *where in flows:
        for p in (a, b):
            if not inside(p):
                ax.text(*p, "☁", ha="center", va="center",
                        fontsize=18, color="grey")
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="-|>", lw=3, color="black",
                                    shrinkA=8, shrinkB=2))
        mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
        ax.add_patch(Circle((mx, my), 0.12, fc="white", ec="black",
                            lw=1.2, zorder=3))
        dx, dy = where[0] if where else (0, -0.3)
        ax.text(mx + dx, my + dy, label, ha="center" if not dx else
                ("left" if dx > 0 else "right"), va="top" if dy < 0 else
                "center", fontsize=8)
    for a, b, label, rad, where in links:
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="->", lw=1, ls="--",
                                    color="#0072B2", shrinkA=4, shrinkB=4,
                                    connectionstyle=f"arc3,rad={rad}"))
        ax.text(*where, label, ha="center", fontsize=8, color="#0072B2")
    ax.set_aspect("equal")
    ax.axis("off")


fig, ax = plt.subplots(figsize=(10, 4.8))
draw_sfd(ax,
         stocks={"Fish stock": (5.2, 4.2), "Boats on\norder": (2.4, 0.6),
                 "New boats": (5.2, 0.6), "Old boats": (8.0, 0.6)},
         flows=[("Regrowth", (2.4, 4.2), (4.35, 4.2)),
                ("Catch", (6.05, 4.2), (8.4, 4.2)),
                ("Ordering", (0.2, 0.6), (1.55, 0.6)),
                ("Building", (3.25, 0.6), (4.35, 0.6)),
                ("Ageing", (6.05, 0.6), (7.15, 0.6)),
                ("Scrapping", (8.85, 0.6), (10.3, 0.6))],
         links=[((5.6, 1.0), (7.1, 3.95), "boats fishing", -0.3,
                 (5.4, 2.3)),
                ((5.7, 3.8), (7.05, 3.95), "density", 0.7, (6.35, 3.15)),
                ((6.9, 3.9), (0.95, 0.95), "catch per boat, price,\n"
                 "profit: orders", 0.2, (2.3, 2.7)),
                ((7.5, 3.9), (9.6, 0.95), "losses: early\nscrapping",
                 -0.25, (9.75, 2.6))])
ax.set_xlim(-0.3, 10.8)
ax.set_ylim(-0.4, 5.0)
ax.set_title("The fishery and its fleet")
plt.show()

Figure: The fishery and its fleet. Thick arrows are flows (fish above; boats below, from order to scrapping through two age classes), and dashed arrows carry information: the boats that go fishing and the density of fish set the catch, the catch per boat, price and profit set new orders, and losses push old boats out early.

**Learning outcomes.** By the end of the chapter you can:

1. **describe** a reference mode from records, and **judge** which signal in the records warns of trouble first (Try it 1);
2. **state** a dynamic hypothesis and a model boundary, starting from an archetype and saying what it lacks (Try it 2);
3. **build** a fishery-and-fleet model step by step from molecules, keeping every policy switched off by default (Try it 3);
4. **test** the model for conservation, extreme conditions and the time step (Try it 4);
5. **explain** why the maximum sustainable yield sits at half the carrying capacity, and which loops and delays make the stock collapse and the fleet outlive it (Try it 5);
6. **compare** and **justify** management policies across an uncertain regrowth rate and an imperfect estimate of the stock (Try it 6).

## 1. The problem and the reference mode

The authority's records from the last cycle hold, for each year, the **landings** (fish brought ashore, in kilotonnes, kt, a year), the number of registered boats, a scientific **survey index** of the stock (an estimate of the stock as a fraction of its unfished size) and the **catch per boat** (tonnes, t, a boat a year), which is the landings divided by the boats. A **reference mode** (chapter 3, From story to diagram) is the pattern over time that the model must explain.

The records below are **synthetic**: the code makes them from a stylised boom-and-bust shape with seeded noise, and they describe no real fishery. We built into them one pattern that fishery scientists report from many fisheries: catch rates that hold up while the stock falls. Replace the generator with real landings, licence and survey data (with their licence and credit line) before using any of this for a real stock.

Two NumPy functions are new. `np.interp(x, xs, ys)` reads a value off a curve given as points, with straight lines between them; it is the same rule as the `TABLE` function of the equation language (chapter 7, Decisions inside the model). `np.clip(a, lo, hi)` keeps values between bounds, and `None` means no bound.

In [ ]:
# Synthetic records of the region's last boom and bust (not real data).
# A stylised shape, seeded: the stock falls along an S-curve, the fleet
# rises and is then slowly scrapped, and fishers' catch per boat holds
# up until the shoals thin out (the pattern fishery scientists report).
yr = np.arange(0, 31)                                  # years since 1
stock_rel = 0.06 + 0.86 / (1 + np.exp((yr - 14) / 2.5))  # of unfished
boats_true = 30 + 330 / (1 + np.exp(-(yr - 10) / 2.2))
boats_true *= np.exp(-np.clip(yr - 17, 0, None) / 10)   # scrapping
SHOAL_X = [0, 0.05, 0.1, 0.2, 0.4, 0.7, 1]   # stock / unfished stock
SHOAL_Y = [0, 0.3, 0.55, 0.78, 0.93, 0.99, 1]  # catch rate / full rate
rate_true = 0.6 * np.interp(stock_rel, SHOAL_X, SHOAL_Y)  # kt per boat
records = pd.DataFrame({
    "landings": boats_true * rate_true * rng.lognormal(0, 0.05, yr.size),
    "boats": np.round(boats_true * rng.lognormal(0, 0.03, yr.size)),
    "survey index": stock_rel * rng.lognormal(0, 0.2, yr.size)},
    index=pd.Index(yr, name="year"))
records["catch per boat"] = 1000 * records["landings"] / records["boats"]
print(records.iloc[::5].round(2).to_string())
print(f"\nlandings peak at {records['landings'].max():.0f} kt/year in "
      f"year {records['landings'].idxmax()}; boats peak at "
      f"{records['boats'].max():.0f} in year {records['boats'].idxmax()}")

In [ ]:
#| fig-alt: "Three panels over 30 years of synthetic records. Left, landings rise to a peak of about 190 kt a year in year 15 and then fall steeply. Middle, the number of boats peaks two years after the landings and declines slowly. Right, the noisy survey index of the stock falls sharply from about year 13, while catch per boat stays near its early level until about year 14 and then declines more slowly."
early = records.loc[0:4].mean()          # average of years 0 to 4
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(12, 3.6))
records["landings"].plot(ax=a1, marker="o", ms=3)
records["boats"].plot(ax=a2, marker="o", ms=3, color="#D55E00", ls="--")
(records["survey index"] / early["survey index"]).plot(
    ax=a3, marker="o", ms=3, label="survey index")
(records["catch per boat"] / early["catch per boat"]).plot(
    ax=a3, label="catch per boat")
a1.set(title="Landings", ylabel="landings (kt/year)")
a2.set(title="Registered boats", ylabel="boats")
a3.set(title="Two signals of the stock", ylabel="index (years 0 to 4 = 1)")
a3.legend(fontsize=8)
for a in (a1, a2, a3):
    a.set_xlabel("year")
plt.tight_layout()
plt.show()

Figure: Synthetic records of the last cycle: landings, registered boats, and two signals of the stock relative to their average in years 0 to 4. Notice that the boats keep increasing after the landings have peaked, and that catch per boat stays near its early level for years after the survey index starts to fall.

Landings rose from 19 kt a year in year 0 to a peak of 189 kt a year in year 15, then fell to 22 kt by year 30. The fleet followed with a lag: from 33 boats to 359 in year 17, two years after the peak in landings, and there were still 98 boats in year 30, landing about an eighth of the peak catch. The survey index is noisy, as surveys are, but its fall from year 13 is clear. Catch per boat, the number every skipper and every fishing company watches, barely moved until year 14.

That is the reference mode the model must reproduce: **overshoot and collapse** of the stock, a **boom and bust** of the fleet that lags the landings, and a commercial signal that lags the stock.

**Try it 1.**

- *Check.* How many years after the peak in landings did the number of boats peak? Read it off the figure, then confirm with code.
- *Apply.* A manager decides to act when catch per boat falls 20% below its average in years 0 to 4. In which year would they act, and what did the survey index read then? Compare with the year the survey index first fell below half.
- *Extend.* Fit a straight line to the logarithm of catch per boat against the logarithm of the survey index, by least squares (chapter 11, Learning from data). What does the slope say about catch per boat as a signal of the stock?

<details><summary>Worked answer</summary>

*Check.* Two years: landings peak in year 15 and boats in year 17.

*Apply.* The survey index is below half its early level in year 13. Catch per boat is not 20% down until year 17, when the survey index read 0.23: the commercial signal sounds when about three quarters of the stock has gone, and four years after the survey.

*Extend.* The slope is 0.34. If catch per boat were proportional to the stock, the slope would be 1. With a slope of about a third, halving the stock cuts catch per boat by only a fifth ($0.5^{0.34} = 0.79$). Fishery scientists call a catch rate that stays stable while the stock declines **hyperstability**. It happens because fish crowd into the shoals that remain and fishers find them there.

</details>

In the code, `first` is a one-line function (a `lambda`, Appendix A) that returns the first year in which a condition holds: `s.index[s]` keeps the years where the boolean Series `s` is `True`.

In [ ]:
# Check: the lag between the peak in landings and the peak in boats
lag = records["boats"].idxmax() - records["landings"].idxmax()
print("boats peak", lag, "years after landings")
# Apply: when would each signal have warned a manager?
rel = records / early
first = lambda s: s.index[s][0]           # first year a condition holds
y_survey = first(rel["survey index"] < 0.5)
y_rate = first(rel["catch per boat"] < 0.8)
print(f"survey index below half: year {y_survey}; catch per boat 20% "
      f"down: year {y_rate}, when the survey index read "
      f"{rel.loc[y_rate, 'survey index']:.2f}")
# Extend: slope of log catch per boat on log survey index (chapter 11)
logs = np.log(records[["catch per boat", "survey index"]])
slope, _ = np.polyfit(logs["survey index"], logs["catch per boat"], 1)
print(f"slope {slope:.2f} (1 would mean catch per boat follows the stock)")

## 2. Dynamic hypothesis and boundary

A **dynamic hypothesis** (chapter 3) names the feedback structure that we think produces the reference mode. Chapter 9 gave us a candidate: **Tragedy of the Commons** (`tragedyCommons`), our own model of the archetype rather than one of Hines' molecules, in which two fleets fish one stock that regrows logistically and each fleet grows while its boats pay. Before building anything new we ask how much of the reference mode the archetype already produces.

In [ ]:
tc = lb.molecule("tragedyCommons")
oa = tc.run(preset="Open access")
fleet = oa["Fleet A"] + oa["Fleet B"]
print(f"Tragedy of the Commons, open access: catch peaks at "
      f"{oa['Catch'].max():.1f} kt/year in year {oa['Catch'].idxmax():.1f};"
      f" boats peak at {fleet.max():.0f} in year {fleet.idxmax():.1f};"
      f" stock lowest {oa['Fish stock'].min():.1f} kt in year "
      f"{oa['Fish stock'].idxmin():.1f}")
print("lag from peak catch to peak fleet:",
      round(fleet.idxmax() - oa["Catch"].idxmax(), 1), "years")
# Catch per boat, relative to its start, when the stock is at half
half = oa.index[oa["Fish stock"] <= 45][0]
print(f"stock halved by year {half}; catch per boat then "
      f"{oa.loc[half, 'Catch per boat'] / oa['Catch per boat'].iloc[0]:.2f}"
      " of its start")

The archetype gets the shape right. The catch peaks at 34.0 kt a year in year 2.6, the boats peak at 87 in year 4.0, 1.4 years later, and the stock falls to 7.5 kt by year 8.1. But it gets the timing and the signal wrong. Everything happens within eight years, because a boat appears the moment profits do and leaves the moment they go. And the warning is perfect: when the stock had halved, in year 3.0, catch per boat had fallen to 0.48 of its start, because each boat catches in proportion to the stock. The records show a fleet that lags by years and a catch per boat that holds up.

So our hypothesis keeps the archetype's core (a commons, fleets that grow on profit) and adds six structures, five of them molecules from earlier chapters. The molecules are after Hines, *Molecules of Structure* [@hines_molecules]; what follows are our own implementations of the structures he describes, written into one model rather than run as separate molecules.

| Structure | What it adds | Molecule (chapter) |
|---|---|---|
| Regrowth | logistic regrowth, fastest at half the carrying capacity | Constrained Growth, `logistic` (3) |
| Boats on order | boats ordered in good years keep arriving after profits fall | the supply line of Stock Management, `stockmgmt` (6) |
| New and old boats | boats last about 20 years; only old boats are scrapped | Aging Chain, `aging` (4) |
| Perceived profit | owners invest on profit as they see it, about a year late | Smooth, `smooth` (4) |
| Price | price eases towards a level set by the landings | Smooth Pricing, `smoothPricing` (6) |
| Shoal effect | catch per boat holds up as fish concentrate in the shoals left | a `TABLE` effect (5) |

The loops these structures make:

| Loop | Path | Polarity | Delays in it |
|---|---|---|---|
| R1 Profits build fleets | fleet, orders, boats on order, fleet (orders are proportional to the fleet) | reinforcing while boats pay | build time |
| B1 Crowding the stock | fleet, catch, fish stock, catch per boat, profit, orders, fleet | balancing | perception, build time, and the shoal effect, which mutes the signal |
| R2 Breeding | fish stock, regrowth, fish stock | reinforcing, dominant when the stock is small | none |
| B2 Limits to regrowth | fish stock, crowding, regrowth, fish stock | balancing, dominant when the stock is large | none |
| B3 Market | catch, price, profit, orders, fleet, catch | balancing in a boom; when fish are scarce the high price keeps boats at sea | price adjustment |
| B4 Exit | losses, early scrapping, fleet | balancing | only old boats leave |

The **boundary** (chapter 3) says what the model explains and what it takes as given:

- **Endogenous**: fish stock, boats on order, new and old boats, price, perceived profit, catch, the share of boats that go to sea; with management switched on, the estimated stock and the quota.
- **Exogenous**: carrying capacity, regrowth rate, costs, the reference price, and the management settings.
- **Excluded**: the age structure of the fish and the years before young fish can be caught, other species and the food web, climate and year-to-year variation in regrowth, where on the fishing grounds boats fish, technology that makes boats more efficient each year, imports that set the price, the labour market and debt.

**Try it 2.**

- *Check.* In Tragedy of the Commons a boat catches 1 kt a year at a full stock of 100 kt, in proportion to the stock, and breaks even at 0.25 kt a year. At what stock does fishing stop paying? Compare it with the stock in year 60 of the open-access run.
- *Apply.* Run the *Slow-growing stock* preset. How low does the stock go, and where is it in year 60?
- *Extend.* Impose a catch limit of 10 kt a year, the most this stock can regrow. Does it save the fish? What stops the fleets growing now?

<details><summary>Worked answer</summary>

*Check.* 0.25 / 1 × 100 = 25 kt, and the run is at 26.1 kt in year 60. Open access settles where fishing just pays, not where the stock regrows fastest (50 kt).

*Apply.* The slow-growing stock falls to 5.2 kt and is back to only 34.5 kt in year 60.

*Extend.* The stock is 53.8 kt in year 60, so the limit saves the fish. But the fleets keep growing until each boat catches only its break-even 0.250 kt a year: 40.0 boats share a catch that half as many could land. The profit the fish could have earned above all costs, the **resource rent**, is spent on extra boats. This is **rent dissipation**, and section 6 meets it again.

</details>

In [ ]:
# Check: the stock at which a boat just breaks even in tragedyCommons
p = tc.preset("Open access")
print("break-even stock:", p["breakEven"] / p["catchFull"] * 100, "kt;",
      "year 60:", round(oa["Fish stock"].iloc[-1], 1), "kt")
# Apply: a slower-growing stock
slow = tc.run(preset="Slow-growing stock")
print(f"slow stock: lowest {slow['Fish stock'].min():.1f} kt, year 60 "
      f"{slow['Fish stock'].iloc[-1]:.1f} kt")
# Extend: a catch limit at the maximum yield, 10 kt/year
lim = tc.run(preset="Open access", catch_limit=10)
f = lim["Fleet A"] + lim["Fleet B"]
print(f"limit 10: stock year 60 {lim['Fish stock'].iloc[-1]:.1f} kt, "
      f"boats {f.iloc[-1]:.1f}, catch per boat "
      f"{lim['Catch per boat'].iloc[-1]:.3f} kt/year")

## 3. The model in code, step by step

We build the model in three steps and run it after each one, so that every new piece is seen working before the next is added.

**Step 1: the stock and the catch, with a fixed fleet.** The stock $B$ (kt) regrows as in Constrained Growth (`logistic`, chapter 5), with a regrowth rate $r$ of 0.3 a year and a carrying capacity $K$ of 1,000 kt:

$$\frac{dB}{dt} = rB\Big(1 - \frac{B}{K}\Big) - \text{Catch}.$$

A boat at sea catches $c_{\max}$ = 600 t a year when the stock is full, scaled by an effect of the density $B/K$. The **shoal effect** $f$ is a `TABLE` (chapter 7): it keeps 93% of the full catch rate at 40% of the stock and 78% at 20%, and falls to zero only as the last shoals go. A weight $s$, `Shoal_concentration`, blends it with the proportional catch of the archetype, so that $s = 0$ gives back catch in proportion to the stock:

$$\text{Potential catch per boat} = c_{\max}\Big[(1 - s)\frac{B}{K} + s\,f\Big(\frac{B}{K}\Big)\Big], \qquad \text{Catch} = \min\Big(\text{Potential catch},\ \frac{B}{\text{Minimum fishing time}}\Big).$$

The minimum fishing time of 0.25 year is the physical limit on an outflow from chapter 1, Why systems push back: even an enormous fleet cannot take the whole stock in an instant.

The code keeps every element in a dictionary keyed by its name. `add` puts an element in, and adding a name that is already there replaces it; that is how step 2 swaps the fixed fleet for a fleet made of stocks without editing step 1. `build` sorts the elements into the stocks, flows and variables of a spec for `lb.Model.from_spec` (chapter 1; Appendix B, @sec-package, documents the spec and the equation language). The `TABLE` text is assembled from the same points the record generator used: `", ".join(...)` glues the pairs together with commas.

In [ ]:
elements = {}   # name -> (kind, element); adding a name again replaces it


def add(kind, i, name, eq, units, src=None, dst=None, floor=True):
    """Add (or replace) a stock, flow or variable of the fishery model."""
    e = {"id": i, "name": name, "units": units}
    if kind == "stock":
        e.update(init=str(eq), nonNegative=floor)
    else:
        e["eq"] = str(eq)
    if src:
        e["from"] = src
    if dst:
        e["to"] = dst
    elements[name] = (kind, e)


def build():
    """A loopbricks model from the elements added so far."""
    group = {k: [e for kind, e in elements.values() if kind == k]
             for k in ("stock", "flow", "var")}
    return lb.Model.from_spec({
        "name": "Fishery and fleet", "timeUnit": "year", "T": 50,
        "dt": 0.0625, "stocks": group["stock"], "flows": group["flow"],
        "vars": group["var"]})


# The shoal effect as a TABLE: x and y points taken in turn
SHOAL = "TABLE(Fish_density, " + ", ".join(
    f"{x}, {y}" for x, y in zip(SHOAL_X, SHOAL_Y)) + ")"

add("stock", "fish", "Fish stock", 900, "kt")
add("flow", "regrow", "Regrowth", "Regrowth_rate * Fish_stock"
    " * (1 - Fish_stock / Carrying_capacity)", "kt/year", dst="fish")
add("flow", "catch", "Catch", "MIN(Potential_catch,"
    " Fish_stock / Minimum_fishing_time)", "kt/year", src="fish")
add("var", "r", "Regrowth rate", 0.3, "1/year")
add("var", "K", "Carrying capacity", 1000, "kt")
add("var", "mft", "Minimum fishing time", 0.25, "year")
add("var", "dens", "Fish density", "Fish_stock / Carrying_capacity",
    "fraction")
add("var", "sc", "Shoal concentration", 1, "fraction")
add("var", "ed", "Effect of density", "(1 - Shoal_concentration)"
    f" * Fish_density + Shoal_concentration * {SHOAL}", "fraction")
add("var", "cmax", "Catch per boat at full stock", 600, "t/boat/year")
add("var", "pcb", "Potential catch per boat",
    "Catch_per_boat_at_full_stock * Effect_of_density", "t/boat/year")
add("var", "fleet", "Fleet", 100, "boats")       # fixed for now
add("var", "pc", "Potential catch",
    "Fleet * Potential_catch_per_boat / 1000", "kt/year")
step1 = build()
rows = {}
for boats in (50, 100, 130, 150, 200, 300):
    d = step1.run(fleet=boats)
    rows[boats] = {"stock, year 50 (kt)": d["Fish stock"].iloc[-1],
                   "catch, year 50 (kt/year)": d["Catch"].iloc[-1]}
print(pd.DataFrame(rows).T.round(1).to_string())

Fifty boats leave 888 kt in the sea after 50 years and land 30 kt a year; 100 boats leave 728 kt and land 60 kt. With 130 boats the catch is 74.9 kt a year from a stock of 552 kt, close to the most this stock can give. But 150 boats, only 15% more, take the stock down to 40 kt by year 50 and are still pushing it down, and 200 or 300 boats wipe it out. A fleet a little too large does not land a little less fish; it lands none. Section 5 explains why.

**Step 2: the fleet, the market and investment.** Boats are ordered into a supply line and delivered after a build time of 2 years, as in the supply line of Stock Management (`stockmgmt`, chapter 8). Delivered boats are new for 10 years, then old for another 10, as in an Aging Chain (`aging`, chapter 6); old boats are scrapped at the end of their life, and earlier when the fleet loses money:

$$\text{Scrapping} = \frac{\text{Old}}{10} + \text{Old} \times 0.3 \times \max(0,\ -\text{Profit signal}).$$

The price $P$ (EUR thousand a tonne) eases towards an indicated price over half a year, as in Smooth Pricing (`smoothPricing`, chapter 8). The indicated price is EUR 1.5 thousand a tonne (EUR 1.50 a kilogram) when 75 kt a year are landed, rises as landings fall with an elasticity of 0.5, and is capped at twice that level:

$$P^{*} = 1.5 \times \min\Big(2,\ \Big(\frac{75}{\text{Catch}}\Big)^{0.5}\Big).$$

A boat costs EUR 200 thousand a year whatever it does (the **fixed cost**: capital, insurance, licences) and EUR 300 thousand a year more if it goes to sea (the **variable cost**: fuel, crew, ice). Boats go to sea only when the catch they expect pays the variable cost, and all go when it pays it one and a half times over: the **utilisation** $u = \min(1, \max(0, 2(cP/\text{VC} - 1)))$, where $c$ is the potential catch per boat. Then

$$\text{Profit per boat} = \text{Catch per boat} \times P - u \times \text{VC} - \text{FC}.$$

Owners act on profit as they perceive it, a first-order smooth of the actual profit over one year (Smooth, `smooth`, chapter 6). The **profit signal** is perceived profit divided by the full cost of a boat at sea (EUR 500 thousand a year). Orders replace the boats that wear out and add a quarter of the profit signal each year:

$$\text{Ordering} = \max\Big(0,\ \text{Fleet}\times\Big(\frac{1}{20} + 0.25 \times \text{Profit signal}\Big)\Big).$$

Owners do not look at the boats already on order. In the language of chapter 8, their supply line weight is zero. The run starts with 900 kt of fish, 20 new and 10 old boats, 4 boats on order and a price of EUR 2.5 thousand a tonne (landings are low, so fish are dear). Printing a model shows its size and its constants.

In [ ]:
# The fleet: a supply line and a two-stage ageing chain
add("stock", "ord", "Boats on order", 4, "boats")
add("stock", "new", "New boats", 20, "boats")
add("stock", "old", "Old boats", 10, "boats")
add("var", "fleet", "Fleet", "New_boats + Old_boats", "boats")
add("flow", "orders", "Ordering", "MAX(0, Desired_orders)", "boats/year",
    dst="ord")
add("flow", "build", "Building", "Boats_on_order / Build_time",
    "boats/year", src="ord", dst="new")
add("flow", "age", "Ageing", "New_boats / Time_as_new", "boats/year",
    src="new", dst="old")
add("flow", "scrap", "Scrapping", "Old_boats / Old_boat_life"
    " + Old_boats * Exit_rate * MAX(0, -Profit_signal)", "boats/year",
    src="old")
add("var", "bt", "Build time", 2, "year")
add("var", "tan", "Time as new", 10, "year")
add("var", "obl", "Old boat life", 10, "year")
add("var", "ex", "Exit rate", 0.3, "1/year")
# The market: price eases towards a price set by the landings
add("stock", "price", "Price", 2.5, "EUR thousand/t")
add("flow", "chp", "Changing price",
    "(Indicated_price - Price) / Price_adjustment_time",
    "EUR thousand/t/year", dst="price")
add("var", "p0", "Reference price", 1.5, "EUR thousand/t")
add("var", "c0", "Reference catch", 75, "kt/year")
add("var", "el", "Price elasticity", 0.5, "dimensionless")
add("var", "ip", "Indicated price", "Reference_price * MIN(2,"
    " (Reference_catch / MAX(1, Catch)) ^ Price_elasticity)",
    "EUR thousand/t")
add("var", "pat", "Price adjustment time", 0.5, "year")
# Fishing pays its way, or boats stay in port
add("var", "vc", "Variable cost", 300, "EUR thousand/boat/year")
add("var", "fc", "Fixed cost", 200, "EUR thousand/boat/year")
add("var", "util", "Utilisation", "MIN(1, MAX(0, 2 * (Potential_catch"
    "_per_boat * Price / Variable_cost - 1)))", "fraction")
add("var", "pc", "Potential catch", "Fleet * Utilisation"
    " * Potential_catch_per_boat / 1000", "kt/year")
add("var", "cpb", "Catch per boat", "1000 * Catch / MAX(1, Fleet)",
    "t/boat/year")
add("var", "ppb", "Profit per boat", "Catch_per_boat * Price"
    " - Utilisation * Variable_cost - Fixed_cost", "EUR thousand/boat/year")
add("var", "prof", "Fleet profit", "Fleet * Profit_per_boat / 1000",
    "EUR million/year")
# Investment follows profit as owners perceive it
add("stock", "pp", "Perceived profit", "Profit_per_boat",
    "EUR thousand/boat/year", floor=False)
add("flow", "perc", "Updating profit",
    "(Profit_per_boat - Perceived_profit) / Perception_time",
    "EUR thousand/boat/year/year", dst="pp")
add("var", "pt", "Perception time", 1, "year")
add("var", "sig", "Profit signal",
    "Perceived_profit / (Variable_cost + Fixed_cost)", "fraction")
add("var", "inv", "Investment sensitivity", 0.25, "1/year")
add("var", "dord", "Desired orders", "Fleet * (1 / (Time_as_new"
    " + Old_boat_life) + Investment_sensitivity * Profit_signal)",
    "boats/year")
step2 = build()
print(step2)

In [ ]:
#| fig-alt: "Four panels over 50 years of the open-access run. The fish stock stays near 900 kt for five years, falls through half the carrying capacity around year 11 and collapses to about 30 kt by year 19, recovering only slowly after that. The fleet grows from 30 to nearly 390 boats, peaking after the catch, and then declines slowly. The catch rises well above the maximum sustainable yield, peaks and collapses. Fleet profit is high at first, then negative from year 12 to year 38."
base2 = step2.run()


def four_panels(runs, title=None):
    """Stock, fleet, catch and fleet profit for a dict of runs."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
    cols = [("Fish stock", "fish stock (kt)"), ("Fleet", "boats"),
            ("Catch", "catch (kt/year)"),
            ("Fleet profit", "fleet profit (EUR million/year)")]
    for ax, (col, unit) in zip(axes.flat, cols):
        for label, d in runs.items():
            d[col].plot(ax=ax, label=label)
        ax.set(title=col, ylabel=unit)
    axes[0, 0].axhline(500, color="grey", lw=1, ls=":")
    axes[1, 0].axhline(75, color="grey", lw=1, ls=":")
    axes[1, 1].axhline(0, color="black", lw=0.8)
    for ax in axes[1]:
        ax.set_xlabel("time (year)")
    if len(runs) > 1:
        axes[0, 0].legend(fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


four_panels({"open access": base2})
print(f"catch peaks at {base2['Catch'].max():.0f} kt/year in year "
      f"{base2['Catch'].idxmax():.1f}; fleet peaks at "
      f"{base2['Fleet'].max():.0f} boats in year "
      f"{base2['Fleet'].idxmax():.1f}")
print(f"stock lowest {base2['Fish stock'].min():.0f} kt in year "
      f"{base2['Fish stock'].idxmin():.1f}; year 50: stock "
      f"{base2['Fish stock'].iloc[-1]:.0f} kt, fleet "
      f"{base2['Fleet'].iloc[-1]:.0f}")
dt = step2.dt
print(f"over 50 years: catch {base2['Catch'].iloc[:-1].sum() * dt:,.0f} kt,"
      f" fleet profit {base2['Fleet profit'].iloc[:-1].sum() * dt:,.0f}"
      " EUR million")
loss = base2.index[base2["Fleet profit"] < 0]     # years with losses
print(f"fleet profit negative from year {loss.min():.1f} to "
      f"{loss.max():.1f}")

Figure: The open-access run over 50 years, with dotted lines at half the carrying capacity (500 kt) and at the maximum sustainable yield (75 kt a year). Notice that the fleet peaks three years after the catch, and that fleet profit stays negative for about a quarter of a century.

The model reproduces the reference mode. The catch peaks at 188 kt a year in year 11.4, two and a half times what the stock can sustain, and the fleet peaks at 388 boats three years later, in year 14.4. The stock collapses to 31 kt, 3% of its unfished size, in year 19.1, and is only 70 kt in year 50. The fleet loses money every year from year 12.1 to year 38.4. Over the 50 years it lands 1,978 kt and loses EUR 150 million in total. The records showed a two-year lag between the peaks; the model gives three. We do not calibrate the model to the records, because they are synthetic and stylised; with real data, chapter 13, Data, calibration and uncertainty, shows how.

**Step 3: management levers, all switched off.** As in chapter 15, every policy is a constant that leaves the model unchanged when it is off:

- **Estimated stock**: managers know the stock only through an assessment. The estimate follows the **measured stock** with an **assessment delay** of 1 year (a first-order smooth again). The measured stock is a blend: with `Survey_share` 1 it is a survey of the stock; with 0 it is inferred from catch rates, $K \times c/c_{\max}$, as if catch per boat were proportional to the stock. `Survey_error` multiplies the measurement; it is 1 (no error) until we feed it a series.
- **Quota**: a **total allowable catch** set each moment as a harvest rate times the estimated stock, when `Quota_on` is 1. The target harvest rate is 0.15 a year, half the regrowth rate (section 5 shows why that is the rate that holds the stock at half its carrying capacity).
- **Harvest control rule**: when `Control_rule_on` is 1, the harvest rate is scaled by a factor that is 1 when the estimated stock is above 50% of $K$, 0 below 20%, and a straight line in between. A **harvest control rule** is a rule, agreed in advance, that cuts fishing as the stock falls.
- **Entry limit**: a `Licence_cap` on the boats in the fleet and on order together; orders stop when it is reached.
- **Policy start**: the levers act from `Policy_start` (year 0 by default), through `STEP(1, Policy_start)` (chapter 9).

The catch now takes the smallest of what the boats can catch, the quota, and the stock divided by the minimum fishing time. `np.allclose`, as in chapter 15, confirms that with every lever off the model gives step 2's run; `base[base2.columns]` picks out the columns that step 2 had.

In [ ]:
# Management levers, all switched off by default
add("stock", "est", "Estimated stock", "Fish_stock", "kt")
add("flow", "assess", "Updating estimate",
    "(Measured_stock - Estimated_stock) / Assessment_delay", "kt/year",
    dst="est")
add("var", "ad", "Assessment delay", 1, "year")
add("var", "se", "Survey error", 1, "dimensionless")
add("var", "ss", "Survey share", 1, "fraction")
add("var", "ms", "Measured stock", "Survey_error * (Survey_share"
    " * Fish_stock + (1 - Survey_share) * Carrying_capacity"
    " * Potential_catch_per_boat / Catch_per_boat_at_full_stock)", "kt")
add("var", "ps", "Policy start", 0, "year")
add("var", "on", "In force", "STEP(1, Policy_start)", "switch")
add("var", "qon", "Quota on", 0, "switch")
add("var", "thr", "Target harvest rate", 0.15, "1/year")
add("var", "hcr", "Control rule on", 0, "switch")
add("var", "blim", "Limit density", 0.2, "fraction")
add("var", "btr", "Trigger density", 0.5, "fraction")
add("var", "ramp", "Rule factor", "MIN(1, MAX(0, (Estimated_stock"
    " / Carrying_capacity - Limit_density)"
    " / (Trigger_density - Limit_density)))", "fraction")
add("var", "hr", "Harvest rate", "Target_harvest_rate * (1"
    " - Control_rule_on + Control_rule_on * Rule_factor)", "1/year")
add("var", "quota", "Quota", "IF(Quota_on * In_force > 0,"
    " Harvest_rate * Estimated_stock, 1000000)", "kt/year")
add("flow", "catch", "Catch", "MIN(Potential_catch, Quota,"
    " Fish_stock / Minimum_fishing_time)", "kt/year", src="fish")
add("var", "cap", "Licence cap", 1000000, "boats")
add("var", "lt", "Licensing time", 1, "year")
add("flow", "orders", "Ordering", "MAX(0, MIN(Desired_orders,"
    " (IF(In_force > 0, Licence_cap, 1000000) - Fleet - Boats_on_order)"
    " / Licensing_time))", "boats/year", dst="ord")
fishery = build()
base = fishery.run()
print("same as step 2 with every lever off:",
      np.allclose(base[base2.columns], base2))
print(fishery)

**Try it 3.**

- *Check.* Work out the orders in year 0 by hand from the profit signal in the run, and compare with `Desired orders`.
- *Apply.* Set `shoal_concentration=0`, so that catch per boat is proportional to the stock. What happens to the stock and the fleet?
- *Extend.* Make boats last 40 years instead of 20 (20 years new, 20 old). How many boats are left in year 30, and where is the stock in year 50?

<details><summary>Worked answer</summary>

*Check.* The profit signal is 1.990 (owners see profits of almost twice the full cost of a boat), so orders are $30 \times (1/20 + 0.25 \times 1.990) = 16.4$ boats a year, as the run says.

*Apply.* Without the shoal effect the stock never falls below 503 kt, the fleet peaks at 308 boats, and the stock settles at 549 kt in year 50. The fishery overshoots a little and corrects itself, because falling catch rates stop investment in time. The shoal effect is the difference between a fishery that corrects itself and one that collapses.

*Extend.* With 40-year boats there are 209 boats in year 30 against 119, and the stock in year 50 is 40 kt against 70. The capital outlives the fish, and keeps fishing the remnant.

</details>

In [ ]:
# Check: the fleet's growth rate in year 0, by hand and from the run
p0 = base.loc[0.0]
print(f"profit signal {p0['Profit signal']:.3f}; desired orders "
      f"{p0['Desired orders']:.1f} boats/year; by hand "
      f"{30 * (1 / 20 + 0.25 * p0['Profit signal']):.1f}")
# Apply: catch per boat proportional to the stock (no shoal effect)
lin = fishery.run(shoal_concentration=0)
print(f"no shoal effect: stock lowest {lin['Fish stock'].min():.0f} kt, "
      f"fleet peak {lin['Fleet'].max():.0f}, stock year 50 "
      f"{lin['Fish stock'].iloc[-1]:.0f} kt")
# Extend: boats that last twice as long
long_life = fishery.run(time_as_new=20, old_boat_life=20)
print(f"40-year boats: fleet in year 30 {long_life.loc[30.0, 'Fleet']:.0f}"
      f" (base {base.loc[30.0, 'Fleet']:.0f}); stock year 50 "
      f"{long_life['Fish stock'].iloc[-1]:.0f} kt")

## 4. Testing the model

Chapter 12 gave four tests that every model should pass before anyone trusts its results: units on every element, conservation of what flows, sensible behaviour under extreme conditions, and an answer that does not depend on the time step. The test below adds up flows with `.iloc[:-1].sum() * dt`, the Euler sum of chapter 1, and reads `attrs["clipped"]`, the report of any floor at zero that changed a stock (chapter 12). `with_init` changes where a stock starts.

In [ ]:
# 1. Units on every element
print("missing units:", [e["name"] for e in fishery.stocks
                         + fishery.flows + fishery.vars
                         if not e.get("units")])
# 2. Conservation of fish and of boats over the open-access run
dt = fishery.dt
net_fish = (base["Regrowth"] - base["Catch"]).iloc[:-1].sum() * dt
change = base["Fish stock"].iloc[-1] - base["Fish stock"].iloc[0]
print(f"fish: net flow {net_fish:.6f} kt, change in stock {change:.6f} kt")
boats = base[["Boats on order", "New boats", "Old boats"]].sum(axis=1)
net_boats = (base["Ordering"] - base["Scrapping"]).iloc[:-1].sum() * dt
print(f"boats: net flow {net_boats:.6f}, change "
      f"{boats.iloc[-1] - boats.iloc[0]:.6f}; floors used: "
      f"{base.attrs['clipped'] or 'none'}")


# 3. Extreme conditions
def extreme(label, model=fishery, **kw):
    """One extreme run: where the stock and the fleet end up."""
    d = model.run(warn_clips=False, **kw)
    return {"test": label, "stock min": d["Fish stock"].min(),
            "stock year 50": d["Fish stock"].iloc[-1],
            "catch max": d["Catch"].max(),
            "fleet year 50": d["Fleet"].iloc[-1],
            "floors": ", ".join(d.attrs["clipped"]) or "none"}


no_boats = (fishery.with_init("New boats", 0).with_init("Old boats", 0)
            .with_init("Boats on order", 0))
huge = fishery.with_init("New boats", 10000)
print(pd.DataFrame([
    extreme("no boats at all", no_boats),
    extreme("no regrowth", regrowth_rate=0),
    extreme("10,000 boats", huge),
    extreme("fish worth nothing", reference_price=0),
]).round(1).to_string(index=False))
# 4. Halve dt
for step in (0.125, 0.0625, 0.03125):
    d = fishery.run(dt=step)
    print(f"dt {step:<7}: stock lowest {d['Fish stock'].min():6.2f} kt"
          f" in year {d['Fish stock'].idxmin():5.2f}; fleet peak "
          f"{d['Fleet'].max():6.1f}")

Every element has units. Over the 50 years, regrowth minus catch adds up to -830.146601 kt, exactly the change in the fish stock, and orders minus scrapping add up to 95.808072 boats, exactly the change in the boats on order, new and old: nothing is created or lost, and no floor at zero was ever needed. The extreme conditions behave as they should:

- **No boats at all**: the stock grows from 900 kt to its carrying capacity of 1,000 kt and nobody ever orders a boat. That is right for this model, where orders are proportional to the fleet, but it shows a limit of the boundary: a real empty fishery with fish in it would attract entrants from outside.
- **No regrowth**: the stock falls to 27.8 kt and stops, because the boats stop going to sea once their catch no longer pays the variable cost; the fleet decays to 11.3 boats.
- **10,000 boats**: the catch hits 3,600 kt a year, the limit of 900 kt divided by the minimum fishing time of 0.25 year, and the stock bottoms out at 27.0 kt without the floor ever being used.
- **Fish worth nothing**: the stock grows to 1,000 kt. The catch of 18.8 kt a year comes in the first months, while the price is still falling from its starting EUR 2.5 thousand a tonne, and the fleet withers to 0.6 boats.

Halving the time step from 0.125 to 0.0625 to 0.03125 year moves the lowest stock from 31.38 to 31.44 to 31.47 kt and the fleet peak from 388.9 to 387.5 to 386.8 boats. Each halving changes the answers by less than 0.5%, so the chosen step of 0.0625 year is fit for this purpose.

**Try it 4.**

- *Check.* In the no-regrowth run, how much fish is caught over the 50 years, and how does it compare with the fall in the stock?
- *Apply.* With 10,000 boats, cut the minimum fishing time from 0.25 to 0.05 year. What does the floor report, and why?
- *Extend.* Run the base model with `method="rk4"` (chapter 12) and compare with Euler's method.

<details><summary>Worked answer</summary>

*Check.* 872.2 kt are caught and the stock falls by 872.2 kt: with no regrowth, every tonne landed is a tonne gone.

*Apply.* The floor holds the fish stock from year 0.1875, the third step. A minimum fishing time of 0.05 year lets the catch reach 20 times the stock a year, so one step of 0.0625 year removes 1.25 times what is there. The step must be shorter than the shortest time constant in the model (chapter 12); the floor report caught the mistake.

*Extend.* Runge-Kutta gives a lowest stock of 31.50 kt against 31.44 and a fleet peak of 386.1 against 387.5. The two methods agree to within half a percent, which confirms the time-step test.

</details>

In [ ]:
# Check: in the no-regrowth run, where did the catch go?
d = fishery.run(regrowth_rate=0)
print(f"no regrowth: catch over 50 years "
      f"{d['Catch'].iloc[:-1].sum() * dt:.1f} kt (stock fell by "
      f"{900 - d['Fish stock'].iloc[-1]:.1f} kt)")
# Apply: a minimum fishing time too short for the time step
for mft in (0.25, 0.05):
    d = huge.run(minimum_fishing_time=mft, warn_clips=False)
    print(f"minimum fishing time {mft}: floors used "
          f"{d.attrs['clipped'] or 'none'}")
# Extend: Euler against Runge-Kutta (chapter 12)
for method in ("euler", "rk4"):
    d = fishery.run(method=method)
    print(f"{method:5}: stock lowest {d['Fish stock'].min():.2f} kt, "
          f"fleet peak {d['Fleet'].max():.1f}")

## 5. Explaining the behaviour

### Why the maximum sustainable yield sits at half the carrying capacity

A stock can be fished for ever at any catch that equals its regrowth. The largest such catch is the **maximum sustainable yield** (MSY). Regrowth $g(B) = rB(1 - B/K)$ is zero when the stock is empty (no fish to breed) and when it is full (crowding cancels breeding), so it peaks in between. Setting its slope to zero,

$$\frac{dg}{dB} = r\Big(1 - \frac{2B}{K}\Big) = 0 \quad\Rightarrow\quad B_{\text{MSY}} = \frac{K}{2}, \qquad \text{MSY} = g\Big(\frac{K}{2}\Big) = \frac{rK}{4}.$$

For our stock that is 500 kt and 75 kt a year. A constant harvest rate $h$ (catch per unit of stock) holds a logistic stock at $K(1 - h/r)$, so the rate that holds it at $K/2$ is $h = r/2 = 0.15$ a year, the target rate of section 3. The code checks the result on the Constrained Growth molecule itself, then draws the regrowth of our fishery against the catch of fleets of different sizes. An equilibrium is where the two curves cross: `np.sign` turns the gap into +1 or -1, `np.diff` finds where the sign changes, and `np.where` returns the positions.

In [ ]:
#| fig-alt: "Left, regrowth of the stock in the Constrained Growth molecule plotted against the population, an arch peaking at half the carrying capacity. Right, for the fishery, the regrowth arch with catch curves for fleets of 100, 130 and 200 boats; with the shoal effect the catch curves are nearly flat until the stock is low, so they cross the arch near the bottom, while a proportional catch curve for 200 boats is a straight line through the origin."
lg = lb.molecule("logistic")
d = lg.run()
net = d["Growth"] - d["Crowding deaths"]          # regrowth, units/year
print(f"Constrained Growth: net growth peaks at {net.max():.2f} a year "
      f"when the population is {d.loc[net.idxmax(), 'Population']:.1f}"
      " (carrying capacity 200)")

B = np.linspace(0, 1000, 2001)                    # stock (kt)
regrowth = 0.3 * B * (1 - B / 1000)               # kt/year
shoal = np.interp(B / 1000, SHOAL_X, SHOAL_Y)


def crossings(catch):
    """Stocks where regrowth equals the catch (equilibria)."""
    gap = regrowth - catch
    i = np.where(np.diff(np.sign(gap)) != 0)[0]
    return np.unique(np.round(B[i[B[i] > 1]]))


fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.plot(d["Population"], net)
a1.axvline(100, color="grey", lw=1, ls=":")
a1.set(title="Constrained Growth: regrowth against size",
       xlabel="population (units)", ylabel="net growth (units/year)")
a2.plot(B, regrowth, color="black", label="regrowth")
for boats in (100, 130, 200):
    catch = boats * 600 * shoal / 1000
    a2.plot(B, catch, label=f"{boats} boats, shoal effect")
    print(f"{boats} boats, shoal effect: equilibria at "
          f"{np.round(crossings(catch)).tolist()} kt")
lin_catch = 200 * 600 * (B / 1000) / 1000
a2.plot(B, lin_catch, label="200 boats, proportional")
print("200 boats, proportional: equilibria at",
      np.round(crossings(lin_catch)).tolist(), "kt")
a2.set(title="Fishery: regrowth and catch at full effort",
       xlabel="fish stock (kt)", ylabel="kt/year", ylim=(0, 130))
a2.legend(fontsize=8)
plt.tight_layout()
plt.show()

Figure: Left, regrowth against size in the Constrained Growth molecule (growth rate 18% a year, carrying capacity 200), peaking at half the carrying capacity. Right, the fishery's regrowth (black) and the catch of fleets at full effort; notice that with the shoal effect the catch curves are almost flat until the stock is low, so they cross the regrowth arch twice or not at all.

The molecule's regrowth peaks at 9.00 a year when the population is 99.6, half its carrying capacity of 200, and $rK/4 = 0.18 \times 200 / 4 = 9$. In the fishery, the shoal effect changes the picture in two ways. With 100 boats the curves cross twice: at 728 kt, a stable equilibrium (above it the catch exceeds regrowth and the stock falls back; below it, it grows back), and at 176 kt, a **tipping point** (chapter 5): below it the catch exceeds regrowth at every stock, and the stock goes to zero. With 130 boats the two crossings have moved together, to 534 and 413 kt. With 200 boats there is no crossing at all: the stock collapses from any start. With a catch proportional to the stock, even 200 boats meet regrowth once, at a stable 600 kt.

That explains step 1. Somewhere between 130 and 150 boats the two equilibria meet and vanish, so a fleet just 15% bigger turns a sustainable fishery into a collapse. Under open access the fleet does not stop at 130 boats, because boats still pay at stocks far below 500 kt.

### Delays: investment that outlives the fish

Why does the fleet overshoot so far? The function below measures the timing of the boom and bust in one run, and the table repeats it with the delays made shorter or longer. `idxmax` returns the time at which a column is largest.

In [ ]:
def story(d):
    """Timing of the boom and bust in one run."""
    turn = d.index[(d.index > 1) & (d["Perceived profit"] < 0)]
    t = turn[0] if len(turn) else np.nan
    return {"catch peak (year)": d["Catch"].idxmax(),
            "fleet peak (year)": d["Fleet"].idxmax(),
            "fleet peak (boats)": d["Fleet"].max(),
            "profit seen < 0 (year)": t,
            "boats on order then": (d.loc[t, "Boats on order"]
                                    if len(turn) else np.nan),
            "stock then (kt)": d.loc[t, "Fish stock"] if len(turn)
            else np.nan,
            "stock lowest (kt)": d["Fish stock"].min(),
            "fleet year 30": d.loc[30.0, "Fleet"]}


variants = {"base": {}, "build in 0.5 years": {"build_time": 0.5},
            "build in 4 years": {"build_time": 4},
            "see profit in 0.25 years": {"perception_time": 0.25},
            "see profit in 3 years": {"perception_time": 3},
            "boats last 10 years": {"time_as_new": 5, "old_boat_life": 5},
            "quick exit (rate 1)": {"exit_rate": 1}}
tab = pd.DataFrame({k: story(fishery.run(**v)) for k, v in
                    variants.items()}).T
print(tab.round(1).to_string())

In the base run the catch peaks in year 11.4. Owners see profit turn negative in year 13.1, when the stock is already down to 197 kt, and at that moment 59.2 boats are still on order. Every one of them is delivered, and the fleet peaks at 388 boats in year 14.4. Orders placed in the good years keep arriving into the bust: the investment outlives the fish.

The delays change the size and length of the bust, not whether it happens. Building in half a year leaves 11.0 boats on order at the turn and 81 boats in year 30; building in four years leaves 124.5 on order and 185 boats in year 30. When owners take three years to see their profits change, the fleet peaks at 613 boats and the turn comes only when the stock is down to 61 kt. Boats that last 10 years rather than 20, or a quick exit when losses mount, shrink the fleet faster afterwards (70 and 87 boats in year 30, against 119). Yet in every row the lowest stock is about 30 kt. The collapse itself comes from the shoal effect and the size the fleet reaches; the delays decide how much capital is stranded and for how long.

### Weak signals

The shoal effect is also why nobody acts in time. The figure compares what a skipper sees, the catch rate of a boat at sea, with what is in the sea.

In [ ]:
#| fig-alt: "Catch rate per boat at sea, with and without the shoal effect, and the fish stock with the shoal effect, each relative to year 0, over the first 25 years. Without the shoal effect the catch rate falls steadily from year 5. With it, the catch rate stays near 1 until about year 11 while the stock falls below half, then plunges after year 13."
fig, ax = plt.subplots()
warn = {}
for label, sc in (("shoal effect", 1), ("proportional", 0)):
    d = fishery.run(shoal_concentration=sc, T=25)
    rate = d["Potential catch per boat"] / d["Potential catch per boat"][0]
    stock = d["Fish stock"] / d["Fish stock"][0]
    rate.plot(ax=ax, label=f"catch rate per boat, {label}")
    first = rate.index[rate < 0.8][0]
    warn[label] = {"catch rate 20% down (year)": first,
                   "stock left then (of start)": stock[first],
                   "fleet then": d.loc[first, "Fleet"],
                   "boats on order then": d.loc[first, "Boats on order"]}
fishery.run(T=25)["Fish stock"].div(900).plot(
    ax=ax, label="fish stock, shoal effect", color="black")
# (without the shoal effect, the catch rate is the stock index itself)
ax.set(title="What the skipper sees and what is in the sea",
       xlabel="time (year)", ylabel="index (year 0 = 1)")
ax.legend(fontsize=8)
plt.show()
print(pd.DataFrame(warn).T.round(2).to_string())

Figure: The catch rate of a boat at sea and the fish stock, each relative to year 0, over the first 25 years. Without the shoal effect the catch rate is the stock index itself; with it, notice that the catch rate stays near 1 until year 11 while the stock has already fallen by more than half.

Without the shoal effect, the catch rate is 20% down by year 9.5, while 80% of the stock remains and the fleet is 240 boats. With it, the catch rate is 20% down only in year 12.9, when 24% of the stock remains, the fleet has 372 boats and 61 more are on order. The same rule of thumb (act when catch rates fall by a fifth) is a timely warning in one world and an obituary in the other. Because the fleet is already built when the signal sounds, the reinforcing loop of profits building fleets has done its work before the balancing loop through the stock can push back.

**Try it 5.**

- *Check.* Compute the MSY, the catch per boat at half the carrying capacity, and the number of boats that would take exactly the MSY there.
- *Apply.* With step 1's model and a fixed fleet of 100 boats, start the stock at 150 kt and at 250 kt, either side of the tipping point at 176 kt, and run for 100 years.
- *Extend.* Halve the investment sensitivity to 0.125. Does a slower boom avoid the collapse?

<details><summary>Worked answer</summary>

*Check.* MSY is 75 kt a year at 500 kt. A boat catches 570 t a year there, so 132 boats would take exactly the MSY, which is why 130 boats sat so close to it in step 1.

*Apply.* From 150 kt the stock goes to 0 by year 100; from 250 kt it grows to 728 kt. The same fleet gives extinction or a healthy stock depending on which side of the tipping point the stock starts.

*Extend.* The fleet peaks lower and later (300 boats in year 21.7), but the stock still collapses, to 32 kt in year 26.2. A slower boom only postpones the bust: 300 boats is still twice the fleet the stock can carry, and the weak signal lets the fleet get there.

</details>

In [ ]:
# Check: maximum sustainable yield and the fleet that takes it
msy = 0.3 * 1000 / 4
rate_half = 600 * np.interp(0.5, SHOAL_X, SHOAL_Y)        # t/boat/year
print(f"MSY {msy:.0f} kt/year at 500 kt; catch per boat there "
      f"{rate_half:.0f} t, so {1000 * msy / rate_half:.0f} boats")
# Apply: the tipping point for 100 boats, tested from both sides
for start in (150, 250):
    d = step1.with_init("Fish stock", start).run(fleet=100, T=100)
    print(f"100 boats from {start} kt: stock year 100 "
          f"{d['Fish stock'].iloc[-1]:.0f} kt")
# Extend: halve the investment sensitivity
d = fishery.run(investment_sensitivity=0.125)
print(f"sensitivity 0.125: fleet peak {d['Fleet'].max():.0f} in year "
      f"{d['Fleet'].idxmax():.1f}, stock lowest "
      f"{d['Fish stock'].min():.0f} kt in year "
      f"{d['Fish stock'].idxmin():.1f}")

## 6. Policies and robustness

The authority can limit the catch, limit the boats, or both. We test five policies, all from year 0 with a survey and a one-year assessment delay:

| Policy | Setting |
|---|---|
| Open access | every lever off |
| Entry limit | a licence cap of 130 boats, the fleet that takes the MSY at half the carrying capacity (Try it 5) |
| Quota, fixed rate | a quota of 0.15 times the estimated stock each year |
| Control rule | the quota, cut when the estimated stock is below 50% of $K$ and closed below 20% |
| Rule and entry limit | the control rule and the licence cap together |

The **scorecard** reports what the authority cares about over 50 years: the stock at the end and at its lowest, the total catch and the catch in the last ten years, the total fleet profit, and the fleet. `d.loc[40:, "Catch"]` selects the rows from year 40 on.

In [ ]:
#| fig-alt: "Four panels over 50 years for three groups of policies. Under open access the stock collapses and the fleet booms and busts. With a quota or the control rule the stock settles at 500 kt and the catch at 75 kt a year, but the fleet grows past 300 boats and settles at 225, and profit falls to about zero. With an entry limit the stock settles near 670 kt, the fleet stays near 115 boats and profit stays near 50 million euros a year."
POLICIES = {
    "open access": {},
    "entry limit": {"licence_cap": 130},
    "quota, fixed rate": {"quota_on": 1},
    "control rule": {"quota_on": 1, "control_rule_on": 1},
    "rule and entry limit": {"quota_on": 1, "control_rule_on": 1,
                             "licence_cap": 130}}


def scorecard(d, dt=0.0625):
    """The decision maker's measures for one 50-year run."""
    return pd.Series({
        "stock y50 (kt)": d["Fish stock"].iloc[-1],
        "stock lowest (kt)": d["Fish stock"].min(),
        "catch 50 y (kt)": d["Catch"].iloc[:-1].sum() * dt,
        "catch y41-50 (kt/y)": d.loc[40:, "Catch"].mean(),
        "profit 50 y (EUR m)": d["Fleet profit"].iloc[:-1].sum() * dt,
        "fleet peak": d["Fleet"].max(),
        "fleet y50": d["Fleet"].iloc[-1]})


runs = {name: fishery.run(**p) for name, p in POLICIES.items()}
# In the central case the rule never binds and the quota never binds
# once entry is limited, so three lines show all five policies
four_panels({"open access": runs["open access"],
             "quota or control rule": runs["control rule"],
             "entry limit, with or without rule": runs["entry limit"]})
score = pd.DataFrame({n: scorecard(d) for n, d in runs.items()}).T
print(score.round(0).astype(int).to_string())

Figure: Fish stock, fleet, catch and fleet profit over 50 years for open access, for a quota (the control rule gives the same run here) and for an entry limit (with or without the rule). Notice that the quota saves the fish but not the profits, while the entry limit saves both.

In the central case the control rule never cuts the quota, because the estimated stock never falls below 50% of $K$, so it gives the same run as the fixed-rate quota; and once entry is limited, the 113 boats never catch the full quota, so the rule adds nothing to the entry limit either. The figure therefore shows three lines for five policies. The table tells the story:

- **Open access** ends with 70 kt of fish, lands 1,978 kt in 50 years (17 kt a year in the last decade) and loses EUR 150 million.
- **A quota at a fixed rate** holds the stock at the MSY level of 500 kt and lands the most, 3,798 kt, 75 kt a year at the end. But it limits the catch, not the boats. While a boat earns more than its costs, owners keep ordering, so the fleet peaks at 319 boats and settles at 225, each catching too little to make money. Over 50 years the fleet earns only EUR 191 million: the rent is dissipated, as in Try it 2.
- **An entry limit** keeps the fleet at 113 to 117 boats, too few to take the full MSY, so the stock settles higher, at 667 kt, and the catch is 3,186 kt, 16% less than with the quota. The fleet earns EUR 2,490 million, thirteen times as much.

Now the stock estimate. The quota is only as good as the estimate it is set from, and estimates are late, noisy, biased, or taken from the wrong signal. The next cell tests the control rule against all four. For the delay test the rule starts only in year 10, when the stock is falling fast and a late estimate matters most. For bias and noise we replace `Survey error` with an input (chapter 1): a function of time `lambda t, b=bias: b` that returns a constant bias (writing `b=bias` fixes each value as the function is made), or a pandas Series with a new random draw for each year. The draws are lognormal with a log mean of $-\sigma^2/2 = -0.045$ for a spread $\sigma$ of 0.3, so that the error averages 1.

In [ ]:
late = {"quota_on": 1, "control_rule_on": 1, "policy_start": 10}
rows = {}
for ad in (0.5, 1, 3, 5):                       # delay in the estimate
    rows[f"rule from year 10, delay {ad}"] = scorecard(
        fishery.run(assessment_delay=ad, **late))
for bias in (0.8, 1.2, 1.5):                    # a constant survey bias
    rows[f"rule, survey bias x{bias}"] = scorecard(fishery.run(
        inputs={"Survey error": lambda t, b=bias: b},
        **POLICIES["control rule"]))
rows["rule, estimate from catch rates"] = scorecard(
    fishery.run(survey_share=0, **POLICIES["control rule"]))
print(pd.DataFrame(rows).T.iloc[:, :5].round(0).astype(int).to_string())

# Random survey error: a new draw each year, 30% spread, 40 replications
years = np.arange(0, 51)
noisy = []
for i in range(40):
    err = pd.Series(rng.lognormal(-0.045, 0.3, years.size), index=years)
    noisy.append(scorecard(fishery.run(inputs={"Survey error": err},
                                       **POLICIES["control rule"])))
noisy = pd.DataFrame(noisy)
print("\ncontrol rule, 30% random survey error, 40 runs (10th, 50th, "
      "90th percentiles):")
print(noisy.quantile([0.1, 0.5, 0.9]).iloc[:, :5].round(0).to_string())

A late estimate costs some fish but no disaster: starting in year 10, an assessment delay of half a year or a year keeps the stock at 500 kt or above, 3 years lets it dip to 459 kt and 5 years to 392 kt, and the catch hardly changes. Random error is a nuisance, not a danger: over 40 runs with a 30% error each year, the median run ends with 525 kt and lands 3,725 kt, close to the error-free run, and even the 10th percentile keeps the lowest stock at 420 kt. The rule smooths the noise because the estimate averages the surveys over the assessment delay.

Systematic error is different. A survey that overestimates by 20% lets the stock settle at 412 kt; one that overestimates by 50% takes it to 316 kt and the catch falls to 3,644 kt. An estimate taken from catch rates is the worst case of all: the shoal effect makes the catch rate say the stock is fine while it collapses, and the rule fails completely, with the stock down to 36 kt, a total catch of 2,137 kt and a loss of EUR 289 million, worse even than open access. An underestimate of 20% costs some catch (3,519 kt) but leaves more fish (613 kt).

### Robustness across the regrowth rate

The regrowth rate is the number we know least: it can be estimated only from how fast the stock has recovered, and it varies with the sea. We cross seven regrowth rates, from 0.15 to 0.45 a year, with the five policies: 35 runs in one call of `run_batch` (chapter 15). The batch output is long, one row per run, time and variable. `groupby` with `.min()` gives each run's lowest value, and `.sum() - .last()` the sum of all rows but the last, the Euler sum used throughout; `.unstack()` turns the variables into columns.

In [ ]:
#| fig-alt: "Two panels against the true regrowth rate from 0.15 to 0.45 a year, one line per policy. Left, the lowest stock in 50 years: open access stays near 30 kt everywhere, the entry limit collapses when regrowth is 0.2 or below, the fixed-rate quota falls low at 0.15, and the control rule stays highest at the low rates. Right, total profit over 50 years: the two policies with an entry limit earn far more than the others when regrowth is 0.25 or above."
grid = [{"regrowth_rate": r, **p} for r in np.arange(0.15, 0.46, 0.05)
        for p in POLICIES.values()]
labels = [(name, round(r, 2)) for r in np.arange(0.15, 0.46, 0.05)
          for name in POLICIES]
settings = pd.DataFrame(grid).fillna(pd.Series(fishery.params))
out = fishery.run_batch(settings, include=["Fish stock", "Catch",
                                           "Fleet profit"])
g = out.groupby(["run", "variable"], observed=True)["value"]
lowest = g.min().unstack()                     # one row per run
total = (g.sum() - g.last()).unstack() * dt    # Euler sum, as before
res = pd.DataFrame({"stock lowest (kt)": lowest["Fish stock"],
                    "catch 50 y (kt)": total["Catch"],
                    "profit 50 y (EUR m)": total["Fleet profit"]})
res.index = pd.MultiIndex.from_tuples([labels[i] for i in res.index],
                                      names=["policy", "regrowth"])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
for name in POLICIES:
    r = res.loc[name]
    a1.plot(r.index, r["stock lowest (kt)"], marker="o", ms=3, label=name)
    a2.plot(r.index, r["profit 50 y (EUR m)"], marker="o", ms=3)
a1.set(title="Lowest stock in 50 years", xlabel="regrowth rate (1/year)",
       ylabel="fish stock (kt)")
a2.set(title="Fleet profit over 50 years", xlabel="regrowth rate (1/year)",
       ylabel="EUR million")
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(res["stock lowest (kt)"].unstack("regrowth").round(0).to_string())
print()
print(res["profit 50 y (EUR m)"].unstack("regrowth").round(0).to_string())
worst = res.groupby(level="policy").min()
print("\nworst case over the seven regrowth rates:")
print(worst.round(0).astype(int).to_string())

Figure: The lowest stock in 50 years (left) and the total fleet profit over 50 years (right), for seven regrowth rates and five policies. Notice that the entry limit alone collapses when regrowth is slow, and that the control rule keeps the lowest stock highest at the low rates.

The entry limit alone fails when regrowth is slow: at 0.15 and 0.2 a year, 130 boats are more than the stock can carry and the lowest stock is 33 and 36 kt. The quota at a fixed rate falls to 109 kt at a regrowth rate of 0.15, because a harvest rate of 0.15 then equals the regrowth rate and $K(1 - h/r) = 0$: the stock is on its way to zero, slowly. The control rule cuts the quota as the stock falls and keeps the lowest stock at 384 kt or more at every rate. The rule and the entry limit together have the best worst case for the stock and for profit, and close to the best for the catch: the lowest stock never below 385 kt, at least 2,177 kt landed (2,186 kt for the rule alone) and at least EUR 1,556 million earned, against 30 kt, 1,358 kt and a loss of EUR 397 million for open access. Neither lever is robust alone; together they are.

**Try it 6.**

- *Check.* Under the quota alone, how much does each boat catch in year 50, at what price, and what does it earn?
- *Apply.* Lower the target harvest rate to a precautionary 0.10 a year in the rule-and-entry-limit policy, and compare the lowest stock and total catch with a rate of 0.15 at regrowth rates of 0.15, 0.3 and 0.45.
- *Extend.* Start the rule and the entry limit in year 10 instead of year 0. Then add a buy-back: use `with_equation` (chapter 9) to scrap 20% of the old boats a year while the fleet is above the cap. How much profit does the late start cost, and how much does the buy-back win back?

<details><summary>Worked answer</summary>

*Check.* 225 boats catch 333 t each, sold at EUR 1.50 thousand a tonne: EUR 500 thousand, exactly the full cost of a boat at sea. Profit per boat is zero (-0.0 in the run). Entry continues until it is.

*Apply.* At a regrowth rate of 0.15, a rate of 0.10 keeps the lowest stock at 448 kt instead of 385 and lands 2,165 kt instead of 2,177, almost the same catch for a much safer stock. At 0.3 and 0.45 nothing changes, because with entry limited the boats cannot catch even the lower quota. Precaution costs little here.

*Extend.* Starting in year 10 cuts the 50-year profit from EUR 2,490 million to 1,447 million: by then the fleet has passed 300 boats and more are on order, and the cap stops new orders but removes nobody. The buy-back wins back part of it, to EUR 1,824 million, by retiring the surplus sooner; both end with 113 boats and the stock never falls below about 500 kt. The cheapest buy-back is the boom that never happened.

</details>

In [ ]:
# Check: catch per boat under the quota alone, in year 50
d = runs["quota, fixed rate"]
print(f"quota alone, year 50: {d['Fleet'].iloc[-1]:.0f} boats, "
      f"{d['Catch per boat'].iloc[-1]:.0f} t each at EUR "
      f"{d['Price'].iloc[-1]:.2f} thousand/t; profit per boat "
      f"{d['Profit per boat'].iloc[-1]:.1f} EUR thousand")
# Apply: a precautionary target harvest rate of 0.1 across the grid
low = [{"regrowth_rate": r, **POLICIES["rule and entry limit"],
        "target_harvest_rate": 0.1} for r in (0.15, 0.3, 0.45)]
for r, row in zip((0.15, 0.3, 0.45), low):
    s = scorecard(fishery.run(**row))
    before = res.loc[("rule and entry limit", r), "catch 50 y (kt)"]
    print(f"r {r}: rate 0.10 gives stock lowest "
          f"{s['stock lowest (kt)']:.0f} kt, catch "
          f"{s['catch 50 y (kt)']:.0f} kt (rate 0.15: {before:.0f} kt)")
# Extend: start in year 10, with and without a buy-back of old boats
buyback = fishery.with_equation(
    "Scrapping", "Old_boats / Old_boat_life + Old_boats * Exit_rate"
    " * MAX(0, -Profit_signal) + 0.2 * Old_boats * In_force"
    " * (Fleet > Licence_cap)")
late_both = dict(POLICIES["rule and entry limit"], policy_start=10)
for label, m in (("no buy-back", fishery), ("buy-back", buyback)):
    s = scorecard(m.run(**late_both))
    print(f"from year 10, {label:<11}: profit {s['profit 50 y (EUR m)']:.0f}"
          f" EUR m, fleet y50 {s['fleet y50']:.0f}, stock lowest "
          f"{s['stock lowest (kt)']:.0f} kt")

## Tell the decision maker

- **Reopening without rules will repeat the last cycle.** In our model of the fishery, open access lands 188 kt in its best year, two and a half times what the stock can sustain, and then the stock collapses to 3% of its unfished size within 20 years. The fleet peaks at 388 boats and loses money for 26 years.
- **Catch per boat is not a warning.** Fish crowd into the shoals that remain, so catches per boat fall by a fifth only when three quarters of the stock has gone. Pay for an independent survey; never set the quota from catch rates (in our runs that collapsed the stock to 36 kt).
- **Limit the boats as well as the catch.** A quota alone keeps the fish but the fleet grows until nobody makes money (EUR 191 million over 50 years). A licence cap of about 130 boats with the quota earns about EUR 2,500 million over the same 50 years.
- **Agree now to cut the quota when the stock is low.** If the stock regrows more slowly than we think, a fixed quota or a cap alone can still collapse it; with a control rule the stock stayed above 380 kt at every regrowth rate we tested.
- **Act at reopening, not when trouble shows.** Waiting ten years loses about EUR 1,000 million of profit, because boats ordered in the boom cannot be un-ordered; a buy-back recovers less than half of it.

The numbers come from a generic model with synthetic records. The ranking of the policies held across every regrowth rate and measurement error we tested; the sizes of the gains will change with real data.

## Check it

Before any policy run leaves your desk, repeat three checks on it, not only on the base run: that units convert where they should (catch per boat is in tonnes and the catch in kilotonnes, so their product with the fleet must carry a factor of 1,000), that no floor at zero was needed, and that the fish still balance.

In [ ]:
# 1. Units: kt and t differ by 1,000, so catch per boat x fleet = catch
d = runs["rule and entry limit"]
gap = (d["Catch per boat"] * d["Fleet"] / 1000 - d["Catch"]).abs().max()
print(f"largest gap between catch per boat x fleet and catch: {gap:.2e}")
# 2. Floors: did any policy run need the floor at zero?
print("floors used:", {n: r.attrs["clipped"] for n, r in runs.items()
                       if r.attrs["clipped"]} or "none")
# 3. Conservation of fish under the policy
net = (d["Regrowth"] - d["Catch"]).iloc[:-1].sum() * dt
print(f"fish books: {net:.6f} against "
      f"{d['Fish stock'].iloc[-1] - d['Fish stock'].iloc[0]:.6f} kt")

The catch per boat times the fleet matches the catch to rounding error, no policy run needed a floor, and under the rule and entry limit the regrowth minus catch matches the change in the stock to six decimal places.

## Going further

- **Age structure.** Real fish take years to reach a size that is caught, so regrowth depends on the stock several years earlier. Replacing the logistic stock with an ageing chain of young and adult fish (chapter 6) adds a recruitment delay that can make even a well-managed stock cycle.
- **Technology creep.** Boats become more efficient each year, often by a few percent. A licence cap on boats then erodes, because each licensed boat catches more; the cap has to be on effort or on catching power.
- **Rights-based management.** Individual transferable quotas give each owner a share of the total catch that can be traded. They attack rent dissipation directly: there is no point in a bigger boat if your share is fixed.
- **Calibration and stock assessment.** With real landings, surveys and fleet data, the regrowth rate, the carrying capacity and the shoal effect can be estimated with the methods of chapter 13, and their uncertainty carried into the policy grid, as in chapter 15.
- **Space.** Boats choose where to fish. A model with several fishing grounds shows how local depletion is hidden by fleets moving on, another source of weak signals.

## Mini-project

The authority asks a second question: what if the region does not reopen carefully, and must rebuild after a collapse? Start the model from the state of the open-access run in year 20 (use `with_init` for every stock, taking the values from `base.loc[20.0]`). Design a rebuilding plan from a moratorium of some years, a buy-back and a control rule with an entry limit afterwards. Report how long the stock takes to reach 500 kt, the total catch and profit over the next 50 years, and how your plan fares across the regrowth rates of section 6. Write the result as a brief of five bullet points, like the one above, and say in the last one which assumption your plan depends on most.

## Further reading

- @moxnes1998 showed in experiments that people overinvest in a fishery even when they own it alone, so the commons is not the only trap; @hardin1968 named the tragedy of the commons and @gordon1954 gave its economics for fisheries.
- @harley2001 found that catch per unit of effort often stays high while the stock falls, the weak signal of section 5; @hilborn1992 is the standard text on estimating fish stocks.
- @sterman2000 treats overshoot and collapse, supply lines and the misperception of feedback in depth, with resource and commodity examples.
- @meadows2008 describes the tragedy of the commons as a system trap, and the ways out: education, privatisation and regulation of the commons.
- @senge1990 introduced the tragedy of the commons as one of the system archetypes for a management audience.
- Hines, *Molecules of Structure* [@hines_molecules], describes the logistic, supply line, ageing chain, smoothing and pricing structures our model is built from.

If you would like to play with the archetype behind this chapter, the LoopBricks app has an interactive version of [Tragedy of the Commons](https://waelrash1.github.io/loopbricks/#/m/tragedyCommons).